In [ ]:
import os
import json
import time
import re
from pathlib import Path
from openai import OpenAI
from dotenv import load_dotenv

# 1. Setup
load_dotenv()
client = OpenAI(
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url="https://api.deepseek.com/v1"
)

def get_smart_chunks(full_text, max_chars=3500):
    """
    Splits a large text into manageable chunks based on topic headers and paragraph breaks.
    
    Args:
        full_text (str): The original text containing "=== TOPIC:" delimiters.
        max_chars (int): The maximum character limit for each chunk.
        
    Returns:    
        list: A list of strings, where each string is a chunk within the char limit.
    """
    # Split the text into segments wherever a new topic starts
    topics = full_text.split("=== TOPIC:")
    final_chunks = []

    for topic_content in topics:
        # Skip empty strings (e.g., if the text starts with a delimiter)
        if not topic_content.strip(): 
            continue
            
        # Re-attach a header (changing "TOPIC" to "THEMA") to the content
        full_topic = "=== THEMA:" + topic_content
        
        # If the entire topic fits within the limit, add it as one chunk
        if len(full_topic) <= max_chars:
            final_chunks.append(full_topic)
        else:
            # If the topic is too long, split it further into paragraphs
            current_chunk = ""
            paragraphs = full_topic.split("\n\n")
            
            for para in paragraphs:
                # Check if adding the next paragraph would exceed the limit
                if len(current_chunk) + len(para) < max_chars:
                    current_chunk += para + "\n\n"
                else:
                    # Save the current chunk and start a new one with the current paragraph
                    if current_chunk:
                        final_chunks.append(current_chunk.strip())
                    current_chunk = para + "\n\n"
            
            # Don't forget to add the last remaining piece
            if current_chunk: 
                final_chunks.append(current_chunk.strip())
                
    return final_chunks

def generate_data(text_chunk, retries=3):
    system_prompt = """You are a Senior AI Training Engineer.  

        Your mission is to produce a GOLD STANDARD SFT (Supervised 
        Fine-Tuning) dataset for a high-performance Arch Linux hyprland zsh AI Assistant. 

        STRICT DATA STRUCTURE: 
        Return a JSON object with a "tasks" key containing an array of 5 objects. 
        Each object MUST contain exactly these three keys: 
        1. "instruction": A clear, diverse, and technically specific Arch Linux, hyprland or zsh task or question use the provided CONTEXT as the anchor. 

        2. "thought": A deep expert-level 'Chain of Thought'. Explain 
            the technical background, weigh alternatives, and show the logical steps
            to the solution. 
        3. "output": The final professional answer, formatted in Markdown with code blocks for commands. 

        CRITICAL JSON RULES (To prevent parsing errors): 
        - ESCAPE all double quotes inside strings using \\" (e.g., "echo \\"hello\\""). 
        - Use \\n for all newlines. NEVER use literal line breaks inside a JSON string. 
        - Ensure every task is a complete, valid JSON object. 

        TECHNICAL FOCUS: 
        - Use the provided CONTEXT as the anchor. 
        - Supplement with expert knowledge for completeness. 
        - Follow the 'Arch Way' (correctness, simplicity, and modern tools)."""

    for attempt in range(retries):
        try:
            response = client.chat.completions.create(
                model="deepseek-v4-pro",
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": f"CONTEXT:\n{text_chunk}"}
                ],
                response_format={ "type": "json_object" }, # force to return a json-obj but without guarantee
                temperature=0.2,
                max_tokens=7000
            )
            
            raw_content = response.choices[0].message.content
            if not raw_content:
                continue
            
            # Step 1: Clean the output. Remove Markdown code blocks (```json ...                
            clean_content = re.sub(r'^```json\s*|\s*```$', '', raw_content.strip(), flags=re.MULTILINE)
            
            try:
            # Step 2: Attempt to parse the cleaned string into a Python dictionary
                data = json.loads(clean_content)
                return data.get("tasks", []), raw_content, True
            except json.JSONDecodeError:
            # Step 3: Fallback - if direct parsing fails, try to find the first JSON object using Regex
                match = re.search(r'(\{.*\})', clean_content, re.DOTALL)
                if match:
                    data = json.loads(match.group(1))
                    return data.get("tasks", []), raw_content, True
                raise

        except Exception as e:
            if attempt == retries - 1: 
                return [], str(raw_content if 'raw_content' in locals() else "Error"), False
            time.sleep(5 * (attempt + 1))
            
    return [], "Timeout/Error", False

def safe_save_json(data, path):
    """
    Saves data to a JSON file atomically to prevent data corruption.
    
    It first writes to a temporary file and then replaces the target file.
    This ensures that if the process crashes during writing, the original 
    file remains intact.
    """
    # Create a temporary path (e.g., data.json.tmp)
    temp_path = path.with_suffix(".tmp")
    
    with open(temp_path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)
    
    # Atomically move/rename the temp file to the final destination
    os.replace(temp_path, path)

def log_raw_output(raw_text, success, chunk_idx):
    """
    Logs every API response to a text file for debugging and auditing.
    
    Args:
        raw_text (str): The raw string received from the LLM.
        success (bool): Whether the content was successfully parsed as JSON.
        chunk_idx (int): The index of the processed text chunk.
    """
    log_path = Path("/workspaces/Arch_PC_Assistent/dataset/raw_responses.txt")
    
    # Determine status label based on parsing success
    status = "SUCCESS (PARSED TO JSON)" if success else "FAILED (RAW TEXT ONLY)"
    
    # Append the log entry with a clear visual separator
    with open(log_path, "a", encoding="utf-8") as f:
        f.write(f"\n{'='*60}\n")
        f.write(f"CHUNK: {chunk_idx} | STATUS: {status} | TIME: {time.ctime()}\n")
        f.write(f"{'='*60}\n")
        f.write(raw_text)
        f.write("\n")

def main():
    base_dir = Path("/workspaces/Arch_PC_Assistent")
    source_file = base_dir / "sources/add_Combined.txt"
    output_path = base_dir / "dataset/add_dataset_progress.json"
    checkpoint_path = base_dir / "dataset/last_chunk_additional.txt"
    output_path.parent.mkdir(parents=True, exist_ok=True)

    all_data = []
    if output_path.exists():
        with open(output_path, "r", encoding="utf-8") as f:
            try:
                all_data = json.load(f)
                print(f"Resuming: Loaded {len(all_data)} entries.")
            except:
                print("Start new dataset.")

    start_index = 0
    if checkpoint_path.exists():
        with open(checkpoint_path, "r") as f:
            try: start_index = int(f.read().strip())
            except: start_index = 0

    with open(source_file, "r", encoding="utf-8") as f:
        chunks = get_smart_chunks(f.read())

    print(f"Arch Factory: {len(chunks)} Chunks total. Start bei {start_index}")
    print("-" * 50)

    for i in range(start_index, len(chunks)):
        chunk = chunks[i]
        title = (chunk.splitlines()[0][:30] + "..") if chunk.splitlines() else "Unknow"
        
        tasks, raw_text, success = generate_data(chunk)
        
        log_raw_output(raw_text, success, i + 1)
        
        if success and tasks:
            all_data.extend(tasks)
            safe_save_json(all_data, output_path)
            status_msg = f"+{len(tasks)} Tasks"
        else:
            status_msg = "Error (Rohdata saved)"

        with open(checkpoint_path, "w") as f:
            f.write(str(i + 1))
        
        progress = (i + 1) / len(chunks)
        bar = "#" * int(progress * 20) + "-" * (20 - int(progress * 20))
        print(f"[{bar}] {progress:>4.1%} | {title:<32} | {status_msg}")
        
        time.sleep(1)

    print("-" * 60)
    print(f"Dataset count: {len(all_data)}")

main()

In [ ]:
file_path = Path("/workspaces/Arch_PC_Assistent/dataset/add_dataset_progress.json")

def validate():
    if not file_path.exists():
        print("Data not found!")
        return

    try:
        with open(file_path, "r", encoding="utf-8") as f:
            data = json.load(f)
        
        print(f"JSON-Syntax is correct. {len(data)} Samples found.")
        
        errors = 0
        for i, item in enumerate(data):
            missing = [key for key in ["instruction", "thought", "output"] if key not in item]
            if missing:
                print(f"Sample {i} missing! Missing Keys: {missing}")
                errors += 1
        
        if errors == 0:
            print("Alle samples are accurate")
        else:
            print(f"{errors} unaccurate sampels found.")

    except json.JSONDecodeError as e:
        print(f"JSON-Syntax error: {e}")

validate()

In [ ]:
input_path = Path("/workspaces/Arch_PC_Assistent/dataset/arch_dataset_progress.json")
output_path = Path("/workspaces/Arch_PC_Assistent/dataset/arch_dataset_cleaned.json")

def fix():
    with open(input_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    
    # Behalte nur Samples, die alle 3 Keys haben
    required_keys = {"instruction", "thought", "output"}
    cleaned_data = [item for item in data if all(k in item for k in required_keys)]
    
    removed = len(data) - len(cleaned_data)
    
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(cleaned_data, f, indent=2, ensure_ascii=False)
    
    print(f"Unaccurate samples cleared.")

fix()

In [ ]:
import os
import json
import time
import re
from pathlib import Path
from openai import OpenAI
from dotenv import load_dotenv

# 1. Setup
load_dotenv()
client = OpenAI(
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url="https://api.deepseek.com/v1"
)

def get_smart_chunks(full_text, max_chars=3500):
    topics = full_text.split("=== THEMA:")
    final_chunks = []
    for topic_content in topics:
        if not topic_content.strip(): continue
        full_topic = "=== THEMA:" + topic_content
        if len(full_topic) <= max_chars:
            final_chunks.append(full_topic)
        else:
            current_chunk = ""
            paragraphs = full_topic.split("\n\n")
            for para in paragraphs:
                if len(current_chunk) + len(para) < max_chars:
                    current_chunk += para + "\n\n"
                else:
                    final_chunks.append(current_chunk.strip())
                    current_chunk = para + "\n\n"
            if current_chunk: final_chunks.append(current_chunk.strip())
    return final_chunks

def generate_troubleshooting_data(text_chunk, retries=3):
    """Spezieller Prompt für Fehlerszenarien und Reparaturen."""
    system_prompt = """You are a Senior Arch Linux Support Engineer (Level 3).
    Your mission is to produce a TROUBLESHOOTING SFT dataset for expert recovery.

    TASK:
    Based on the CONTEXT, create 3 unique scenarios where something fails (errors, boot loops, driver crashes, misconfigurations).

    STRUCTURE:
    Return a JSON object with a "tasks" key. Each task MUST have:
    1. "instruction": A realistic user report or error message (e.g., 'My screen is black after NVIDIA update' or 'UUID mismatch on boot').
    2. "thought": A deep diagnostic analysis. Identify the root cause based on the context, mention relevant logs (journalctl, dmesg), and plan the recovery.
    3. "output": A professional, step-by-step recovery guide using terminal commands and Arch best practices.

    CRITICAL RULES:
    - Focus on 'What can go wrong' in this specific context.
    - ESCAPE all double quotes with \\". 
    - Use \\n for newlines. 
    - Respond ONLY with the JSON object."""

    for attempt in range(retries):
        try:
            response = client.chat.completions.create(
                model="deepseek-v4-pro",
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": f"CONTEXT:\n{text_chunk}"}
                ],
                response_format={ "type": "json_object" },
                temperature=0.3,
                max_tokens=5000 
            )
            
            raw_content = response.choices[0].message.content
            clean_content = re.sub(r'^```json\s*|\s*```$', '', raw_content.strip(), flags=re.MULTILINE)
            
            data = json.loads(clean_content)
            return data.get("tasks", [])
            
        except Exception:
            time.sleep(5 * (attempt + 1))
            
    return []

def main():
    base_dir = Path("/workspaces/Arch_PC_Assistent")
    source_file = base_dir / "sources/Arch_Wiki_Combined.txt"
    output_path = base_dir / "dataset/arch_troubleshooting_dataset.json"
    checkpoint_path = base_dir / "dataset/last_chunk_troubleshoot.txt"
    output_path.parent.mkdir(parents=True, exist_ok=True)

    all_data = []
    seen_instructions = set()
    if output_path.exists():
        with open(output_path, "r", encoding="utf-8") as f:
            try:
                all_data = json.load(f)
                for item in all_data:
                    if "instruction" in item:
                        seen_instructions.add(item["instruction"].strip().lower())
            except: pass

    start_index = 0
    if checkpoint_path.exists():
        with open(checkpoint_path, "r") as f:
            try: 
                start_index = int(f.read().strip())
            except: 
                start_index = 0

    if not source_file.exists():
        print(f"Error: sourcedata {source_file} not found.")
        return

    with open(source_file, "r", encoding="utf-8") as f:
        chunks = get_smart_chunks(f.read())

    print(f"Arch Troubleshooting Factory - Start bei Chunk {start_index}/{len(chunks)}")
    print("-" * 60)

    for i in range(start_index, len(chunks)):
        chunk = chunks[i]
        title_line = chunk.splitlines()[0] if chunk.splitlines() else "Unknow"
        title = (title_line[:35] + '..') if len(title_line) > 35 else title_line
        
        start_time = time.time()
        results = generate_troubleshooting_data(chunk)
        
        if results:
            new_entries = 0
            for task in results:
                if isinstance(task, dict) and "instruction" in task:
                    instr_clean = str(task["instruction"]).strip().lower()
                    if instr_clean not in seen_instructions:
                        all_data.append(task)
                        seen_instructions.add(instr_clean)
                        new_entries += 1
            
            with open(output_path, "w", encoding="utf-8") as f:
                json.dump(all_data, f, indent=2, ensure_ascii=False)
            with open(checkpoint_path, "w") as f:
                f.write(str(i + 1))
            
            progress = (i + 1) / len(chunks)
            print(f"[{progress:>4.1%}] {title:<37} | +{new_entries} Fixes")
        else:
            print(f"[{i+1}/{len(chunks)}] Warning: error with Chunk '{title}'")
        
        time.sleep(1)

    print(f"Phase 2 DONE! Troubleshooting-Eintraege counts: {len(all_data)}")

main()